
# BUS5001 Assessment 3 – Question 3
## ESG Message Triage Using an Open-Source Large Language Model

### Project Overview

This notebook investigates the use of an open-source Large Language Model (LLM) for classifying and prioritising organisational Environmental, Social and Governance (ESG) incidents.

The experiment examines how prompt engineering influences classification accuracy, response structure and escalation decisions.

### Objectives

1. Implement an open-source LLM for ESG incident classification.
2. Evaluate a basic ESG classification prompt.
3. Improve the prompt using explicit classification criteria and structured JSON output.
4. Compare the model's responses across three ESG incident scenarios.
5. Compare the LLM with rule-based and zero-shot classification approaches.
6. Discuss limitations relating to accuracy, hallucinations, privacy, escalation and human oversight.

### Technologies

| Technology | Purpose |
|---|---|
| Google Colab | Experiment development and execution |
| Python | Data processing and evaluation |
| NVIDIA Tesla T4 | GPU acceleration |
| Hugging Face Transformers | Pretrained model integration |
| Qwen2.5-1.5B-Instruct | Primary open-source LLM |
| facebook/bart-large-mnli | Zero-shot ESG classification baseline implemented and evaluated |
| Rule-based classifier | Keyword-based ESG classification baseline implemented and evaluated |
| GitHub | Experiment documentation and evidence |

### Methodology

The experiment follows five stages:

1. Environment configuration and model loading.
2. Original prompt evaluation.
3. Improved prompt evaluation.
4. Baseline classifier comparison.
5. Results analysis and ethical evaluation.

Three fictional ESG incidents are used as a consistent test dataset.

### Ethical Considerations

All test scenarios are fictional and do not contain personal or confidential organisational information.

The LLM is evaluated as a decision-support tool rather than an autonomous decision-maker. Its classifications and escalation recommendations would require appropriate human oversight in real-world applications.

### Reproducibility

The notebook records the model identifier, library versions, computational environment, prompts, test messages and generated responses.

Results are reported as observed without manually correcting model predictions.



## Experiment 1: Environment Setup and Verification

### Objective

Prepare the Google Colab environment and verify that the required machine learning libraries and GPU resources are available.

### Implementation

The environment uses Python, PyTorch, Hugging Face Transformers and Accelerate.

GPU availability is checked to confirm that the pretrained language model can perform inference using hardware acceleration.

### Observed Results

The environment verification returned:

| Component | Result |
|---|---|
| PyTorch | 2.11.0+cu130 |
| Transformers | 4.57.6 |
| Accelerate | 1.15.0 |
| Hugging Face Hub | 0.36.2 |
| GPU available | True |
| GPU device | NVIDIA Tesla T4 |

### Observation

The required libraries imported successfully, and GPU acceleration was available.

Package dependency warnings were reported during installation, but they did not prevent the subsequent model-loading and inference experiments.


In [19]:

%pip install -q "transformers>=4.45,<5" accelerate sentencepiece


In [20]:

# ============================================================
# Experiment 1: Environment and GPU Verification
# ============================================================

# Purpose:
# Verify that the required machine learning libraries are
# available and that Google Colab can access GPU acceleration.
#
# This check helps establish the computational environment
# used to run and evaluate the open-source language model.

import torch
import transformers
import accelerate
import huggingface_hub

# Record library versions for experiment reproducibility.
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

# Check whether GPU acceleration is available.
gpu_available = torch.cuda.is_available()

print("GPU available:", gpu_available)

if gpu_available:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Model execution may be slower.")


PyTorch: 2.11.0+cu130
Transformers: 4.57.6
Accelerate: 1.15.0
Hugging Face Hub: 0.36.2
GPU available: True
GPU: Tesla T4



## Experiment 2: Open-Source LLM Configuration

### Objective

Load a pretrained instruction-following language model suitable for ESG message classification.

### Model Selection

**Model:** Qwen2.5-1.5B-Instruct

**Hugging Face identifier:** `Qwen/Qwen2.5-1.5B-Instruct`

The model was selected because its relatively small size makes it practical to execute in Google Colab while supporting instruction-based text generation.

### Implementation

The model and tokenizer are loaded using Hugging Face Transformers.

Half-precision floating-point computation (`float16`) is used to reduce GPU memory requirements.

The model is configured for inference rather than training.

### Observed Results

| Configuration | Result |
|---|---|
| Model | Qwen2.5-1.5B-Instruct |
| Execution device | cuda:0 |
| Numerical precision | torch.float16 |
| Model loading | Successful |

### Observation

The model loaded successfully on the Tesla T4 GPU.

This confirms that the selected open-source model can be used for the subsequent ESG classification experiments.


In [21]:

# ============================================================
# Experiment 2: Load the Open-Source Language Model
# ============================================================

# Objective:
# Load Qwen2.5-1.5B-Instruct from Hugging Face to support
# automated classification of organisational ESG messages.
#
# The model will later be evaluated using both the original
# assignment prompt and a revised, more structured prompt.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Record the exact model identifier for reproducibility.
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Load the tokenizer used to convert text into model tokens.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the pretrained model using half-precision GPU computation.
# device_map="auto" places the model on available hardware.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Set evaluation mode for inference rather than model training.
model.eval()

print("Model loaded successfully:", MODEL_ID)
print("Model device:", model.device)
print("Model precision:", next(model.parameters()).dtype)


Model loaded successfully: Qwen/Qwen2.5-1.5B-Instruct
Model device: cuda:0
Model precision: torch.float16



## Experiment 3: Initial Model Inference Test

### Objective

Verify that the loaded language model can interpret an ESG-related incident and generate a relevant classification response.

This is a functionality test conducted before the formal prompt comparison.

### Test Scenario

A factory employee reports that untreated wastewater is being discharged into a nearby river.

**Expected ESG category:** Environmental

### Observed Result

The model identified the incident as an environmental concern and explained that untreated wastewater could pollute water sources and harm ecosystems.

### Observation

The model successfully generated a relevant response and recognised the environmental nature of the incident.

However, this single test does not establish overall classification accuracy.

The subsequent experiments evaluate multiple ESG categories using consistent test messages.


In [22]:

# ============================================================
# Experiment 3: Initial Model Inference Test
# ============================================================

# Objective:
# Verify that the open-source LLM can process an ESG-related
# message and generate a response using GPU inference.
#
# This is a functionality test, not the formal prompt evaluation.

test_message = (
    "A factory employee reports that untreated wastewater "
    "is being discharged into a nearby river."
)

messages = [
    {
        "role": "system",
        "content": "You are an assistant specialising in ESG issues."
    },
    {
        "role": "user",
        "content": (
            "Identify the ESG category of this message "
            "and briefly explain your reasoning:\n"
            + test_message
        )
    }
]

# Format messages using the model's instruction template.
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert the prompt into tensors on the model's device.
inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(model.device)

# Generate a deterministic response for reproducibility.
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only newly generated tokens.
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("ESG Message:", test_message)
print("\nModel Response:\n", response)


ESG Message: A factory employee reports that untreated wastewater is being discharged into a nearby river.

Model Response:
 The ESG category for this message would be Environmental Impact.

Reasoning: The issue described involves the discharge of untreated wastewater, which directly impacts the environment by polluting water sources like rivers. This can have significant negative effects on aquatic ecosystems, human health, and overall environmental sustainability. Therefore, addressing this problem falls under the broader umbrella of Environmental Impact concerns.



## Experiment 4: ESG Test Dataset

### Objective

Create a consistent dataset for evaluating the classification performance of different prompts and models.

### Dataset Design

Three fictional incidents were developed to represent the primary ESG categories.

| Incident ID | Scenario | Reference Category |
|---|---|---|
| ESG001 | Untreated wastewater discharged into a river | Environmental |
| ESG002 | Unsafe warehouse conditions and excessive overtime | Social |
| ESG003 | Undisclosed conflict of interest involving supplier payments | Governance |

### Methodology

Each incident is assigned a reference category based on its primary ESG concern.

The same incident descriptions will be used for:

- Original prompt evaluation
- Improved prompt evaluation
- Rule-based classification
- Zero-shot classification

Using identical test cases supports a controlled comparison between classification approaches.

### Limitations

The dataset contains only three synthetic incidents.

The results provide an illustrative comparison but are insufficient to establish general model accuracy or reliability across real organisational incidents.


In [23]:

# ============================================================
# Experiment 4: ESG Test Dataset
# ============================================================

# Three fictional incidents representing different ESG categories.
# The same messages will be used across all classification methods.

esg_test_data = [
    {
        "id": "ESG001",
        "message": (
            "A factory employee reports that untreated wastewater "
            "is being discharged into a nearby river."
        ),
        "expected_category": "Environmental"
    },
    {
        "id": "ESG002",
        "message": (
            "Several warehouse employees report unsafe working "
            "conditions, excessive overtime and a lack of "
            "appropriate protective equipment."
        ),
        "expected_category": "Social"
    },
    {
        "id": "ESG003",
        "message": (
            "An employee reports that a senior manager approved "
            "payments to a supplier owned by a family member "
            "without declaring the conflict of interest."
        ),
        "expected_category": "Governance"
    }
]

# Display the test cases and their expected classifications.
for incident in esg_test_data:
    print(f"\nIncident: {incident['id']}")
    print(f"Message: {incident['message']}")
    print(f"Expected category: {incident['expected_category']}")



Incident: ESG001
Message: A factory employee reports that untreated wastewater is being discharged into a nearby river.
Expected category: Environmental

Incident: ESG002
Message: Several warehouse employees report unsafe working conditions, excessive overtime and a lack of appropriate protective equipment.
Expected category: Social

Incident: ESG003
Message: An employee reports that a senior manager approved payments to a supplier owned by a family member without declaring the conflict of interest.
Expected category: Governance



## Experiment 5: Basic Prompt Evaluation

### Objective

Evaluate the model's initial ESG classification and prioritisation performance using a basic natural-language prompt.

### Prompt Design

The prompt instructs the model to:

1. Identify the primary ESG category.
2. Assign a Low, Medium or High priority.
3. Provide a brief explanation.

The prompt does not include detailed category definitions, a JSON schema or explicit human-escalation criteria.

### Results

| Incident | Reference Category | Predicted Category | Priority | Correct |
|---|---|---|---|---|
| ESG001 | Environmental | Environmental | High | Yes |
| ESG002 | Social | Social | High | Yes |
| ESG003 | Governance | Social | High | No |

**Classification accuracy: 2/3 = 66.7%**

### Analysis

The model correctly identified the environmental pollution and workplace safety incidents.

However, it classified the undisclosed supplier conflict of interest as Social rather than Governance.

Although the response recognised ethical concerns and conflicts of interest, it assigned the incident to the wrong ESG category.

This indicates that the basic prompt did not provide sufficient guidance to distinguish governance misconduct from broader social or ethical concerns.

### Limitations

The model produced natural-language responses rather than structured JSON.

The prompt also lacked explicit escalation criteria, limiting its suitability for automated incident triage.

### Improvement Opportunity

The next experiment introduces clearer ESG definitions, a standardised output schema and explicit priority and escalation instructions.

The basic prompt provides an initial comparison condition for evaluating the effects of more explicit classification instructions and structured output requirements.


In [24]:

# ============================================================
# Experiment 5: Basic Prompt Evaluation
# ============================================================

# Basic prompt used as the initial comparison condition.
# No JSON schema or explicit escalation criteria are provided.

original_prompt = """
You are an ESG classification assistant.

Read the following organisational message.
Identify its ESG category (Environmental, Social, or Governance),
assign a priority level (Low, Medium, or High),
and briefly explain your decision.

Message:
{message}
"""

original_results = []

for incident in esg_test_data:

    messages = [
        {
            "role": "user",
            "content": original_prompt.format(
                message=incident["message"]
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    # Preserve the original model output for later comparison.
    original_results.append({
        "id": incident["id"],
        "expected_category": incident["expected_category"],
        "model_response": response
    })

    print(f"\n{'=' * 60}")
    print(f"Incident: {incident['id']}")
    print(f"Expected category: {incident['expected_category']}")
    print(f"\nQwen Response:\n{response}")



Incident: ESG001
Expected category: Environmental

Qwen Response:
**ESG Category:** Environmental

**Priority Level:** High

**Explanation:**
The issue of untreated wastewater discharging into a nearby river falls under the Environmental category because it directly impacts the environmental health and safety of the surrounding ecosystem. The discharge can lead to water pollution, harm aquatic life, and potentially affect human health through contaminated drinking water sources. Given the severity of this problem, which involves immediate environmental damage without any indication of mitigation efforts, the priority level is set as High.

Incident: ESG002
Expected category: Social

Qwen Response:
**ESG Category:** Social

**Priority Level:** High

**Explanation:**
The message highlights issues related to employee safety, work-life balance, and protection against hazards at the workplace. These concerns fall under the Social pillar as they directly impact the well-being and rights of 


## Experiment 6: Improved Prompt Evaluation

### Objective

Investigate whether a more structured prompt improves ESG classification accuracy and output consistency.

### Prompt Engineering Improvements

The revised prompt introduces:

1. Explicit definitions for Environmental, Social and Governance categories.
2. Clear priority criteria for Low, Medium and High incidents.
3. Human-escalation instructions for serious incidents.
4. Restrictions against inventing facts or treating allegations as verified.
5. A required JSON response structure.

### Expected JSON Fields

| Field | Purpose |
|---|---|
| category | Primary ESG classification |
| priority | Incident severity |
| escalate_to_human | Whether human review is recommended |
| reason | Brief explanation supporting the decision |

### Results

| Incident | Reference Category | Predicted Category | Priority | Escalation | Correct |
|---|---|---|---|---|---|
| ESG001 | Environmental | Environmental | High | True | Yes |
| ESG002 | Social | Social | High | True | Yes |
| ESG003 | Governance | Governance | High | True | Yes |

**Classification accuracy: 3/3 = 100%**

### Comparison

| Metric | Original Prompt | Improved Prompt |
|---|---|---|
| Correct classifications | 2/3 | 3/3 |
| Classification accuracy | 66.7% | 100% |
| Explicit escalation decision | No | Yes |
| Structured response fields | No | Yes |

### Analysis

The improved prompt correctly classified all three test incidents.

The most important change occurred in ESG003, where the model correctly identified an undisclosed conflict of interest as a Governance issue.

The revised prompt also produced explicit priority and human-escalation decisions for every incident.

All three incidents received High priority and were recommended for human escalation.

### Observed Limitation

Although the prompt requested JSON without Markdown formatting, the model wrapped each response in a Markdown JSON code block.

The JSON objects appear structurally consistent, but the raw responses require validation before they can be used in an automated processing pipeline.

### Interpretation

The results suggest that explicit ESG definitions and structured instructions improved classification performance on this small dataset.

However, the 100% result applies only to these three examples and does not demonstrate general reliability.

Additional testing would be required before deployment in a real organisational environment.


In [25]:

# ============================================================
# Experiment 6: Improved ESG Classification Prompt
# ============================================================

# This experiment evaluates whether explicit ESG definitions,
# priority criteria and structured JSON improve classification.

import torch

improved_prompt = """
You are an ESG incident triage assistant.

Classify the organisational incident using these definitions:

Environmental:
Pollution, emissions, waste, water contamination,
resource consumption or ecological damage.

Social:
Employee safety, labour rights, discrimination,
human rights, workplace conditions or community welfare.

Governance:
Corruption, bribery, conflicts of interest, fraud,
undisclosed related-party transactions, unethical procurement
or failures of corporate oversight.

INSTRUCTIONS:
1. Select exactly one primary ESG category.
2. Determine priority:
   - High: Serious harm, immediate safety or environmental
     risks, or significant suspected misconduct.
   - Medium: Material concerns requiring investigation
     without clear immediate danger.
   - Low: Minor issues without significant immediate risk.
3. Set escalate_to_human to true when the incident involves
   serious harm, suspected misconduct or significant risk.
4. Base the classification only on the supplied message.
5. Do not invent facts or assume the allegation is verified.
6. Return only valid JSON, without Markdown formatting.

Required JSON structure:
{
  "category": "Environmental | Social | Governance",
  "priority": "Low | Medium | High",
  "escalate_to_human": true,
  "reason": "Brief evidence-based explanation"
}

Incident:
{message}
"""

# Store raw outputs for comparison with the original prompt.
improved_results = []

for incident in esg_test_data:

    # Replace only the message placeholder.
    # This preserves the JSON braces in the prompt.
    user_prompt = improved_prompt.replace(
        "{message}",
        incident["message"]
    )

    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # Apply the model's chat template and generate an
    # explicit attention mask for reliable inference.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # Deterministic generation supports reproducibility.
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated response.
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Preserve actual model outputs without manual correction.
    improved_results.append({
        "id": incident["id"],
        "message": incident["message"],
        "expected_category": incident["expected_category"],
        "model_response": response
    })

    print(f"\n{'=' * 60}")
    print(f"Incident: {incident['id']}")
    print(f"Expected category: {incident['expected_category']}")
    print(f"\nImproved Prompt Response:\n{response}")



Incident: ESG001
Expected category: Environmental

Improved Prompt Response:
```json
{
  "category": "Environmental",
  "priority": "High",
  "escalate_to_human": true,
  "reason": "The discharge of untreated wastewater into a nearby river poses severe environmental harm and immediate safety risks."
}
```

Incident: ESG002
Expected category: Social

Improved Prompt Response:
```json
{
  "category": "Social",
  "priority": "High",
  "escalate_to_human": true,
  "reason": "The reported unsafe working conditions, excessive overtime, and lack of appropriate protective equipment indicate severe material concerns that require immediate attention."
}
```

Incident: ESG003
Expected category: Governance

Improved Prompt Response:
```json
{
  "category": "Governance",
  "priority": "High",
  "escalate_to_human": true,
  "reason": "The report indicates potential corruption and undisclosed related-party transactions, which could lead to significant ethical violations."
}
```



## Experiment 7: JSON Output Validation

### Objective

Evaluate whether the improved prompt produces machine-readable responses suitable for automated ESG incident processing.

### Validation Criteria

Each model response is evaluated against the following requirements:

1. The raw response must be valid JSON.
2. All four required fields must be present.
3. The ESG category must contain an allowed value.
4. Priority must be Low, Medium or High.
5. The escalation field must contain a Boolean value.
6. The explanation must contain non-empty text.

### Validation Approach

Two forms of validation are performed:

**Raw JSON validation**

Checks whether the model's response can be parsed directly without modification.

**Schema validation after formatting cleanup**

Removes optional Markdown code fences before parsing and checks the required fields and allowed values.

This distinction is important because a response can contain a valid JSON object without being directly machine-readable as raw JSON.

### Evaluation Measures

The experiment records:

- Number of directly parseable JSON responses.
- Number of responses passing schema validation after cleanup.
- Number of correctly classified incidents.


### Results

| Metric | Result |
|---|---|
| Correct classifications | 3/3 (100%) |
| Valid raw JSON | 0/3 (0%) |
| Valid JSON after cleanup | 3/3 (100%) |
| Schema validation after cleanup | 3/3 (100%) |

### Discussion

All three responses contained correctly classified ESG incidents and passed schema validation after Markdown formatting was removed.

However, none of the raw responses could be parsed directly as JSON because the model included Markdown code fences despite instructions to return JSON only.

This demonstrates that prompt instructions alone cannot guarantee strict output formatting.

For practical deployment, the classification pipeline should include response preprocessing, JSON parsing, schema validation and error handling before forwarding results to downstream applications.

The experiment also demonstrates the distinction between classification accuracy and output-format reliability.

Although classification accuracy reached 100% on the three test incidents, this small dataset is insufficient to establish general model reliability.



In [26]:

# ============================================================
# Experiment 7: JSON Output Validation
# ============================================================

# Evaluate structural reliability without modifying the
# original responses stored in improved_results.

import json
import re

required_fields = {
    "category", "priority", "escalate_to_human", "reason"
}

valid_categories = {"Environmental", "Social", "Governance"}
valid_priorities = {"Low", "Medium", "High"}

validation_results = []

for result in improved_results:
    raw_response = result["model_response"].strip()

    # Check whether the response is directly valid JSON.
    try:
        json.loads(raw_response)
        raw_json_valid = True
    except json.JSONDecodeError:
        raw_json_valid = False

    # Remove optional Markdown fences for parsing.
    cleaned_response = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw_response
    ).strip()

    try:
        parsed = json.loads(cleaned_response)

        schema_valid = (
            isinstance(parsed, dict)
            and set(parsed.keys()) == required_fields
            and parsed["category"] in valid_categories
            and parsed["priority"] in valid_priorities
            and type(parsed["escalate_to_human"]) is bool
            and isinstance(parsed["reason"], str)
            and bool(parsed["reason"].strip())
        )

        category_correct = (
            schema_valid
            and parsed["category"] == result["expected_category"]
        )

        validation_status = "PASS" if schema_valid else "INVALID SCHEMA"

    except json.JSONDecodeError:
        parsed = None
        schema_valid = False
        category_correct = False
        validation_status = "INVALID JSON"

    validation_results.append({
        "id": result["id"],
        "raw_json_valid": raw_json_valid,
        "schema_valid": schema_valid,
        "category_correct": category_correct,
        "status": validation_status
    })

    print(f"\nIncident: {result['id']}")
    print(f"Raw JSON valid: {raw_json_valid}")
    print(f"Schema valid after cleaning: {schema_valid}")
    print(f"Category correct: {category_correct}")
    print(f"Validation status: {validation_status}")

print("\n--- Evaluation Summary ---")
print(
    "Correct classifications:",
    sum(r["category_correct"] for r in validation_results),
    "/",
    len(validation_results)
)
print(
    "Valid raw JSON:",
    sum(r["raw_json_valid"] for r in validation_results),
    "/",
    len(validation_results)
)



Incident: ESG001
Raw JSON valid: False
Schema valid after cleaning: True
Category correct: True
Validation status: PASS

Incident: ESG002
Raw JSON valid: False
Schema valid after cleaning: True
Category correct: True
Validation status: PASS

Incident: ESG003
Raw JSON valid: False
Schema valid after cleaning: True
Category correct: True
Validation status: PASS

--- Evaluation Summary ---
Correct classifications: 3 / 3
Valid raw JSON: 0 / 3



## Experiment 8: Rule-Based ESG Classification

### Objective

Implement a traditional keyword-based ESG classifier as a baseline for comparison with the open-source LLM.

### Methodology

The classifier uses predefined keyword lists representing Environmental, Social and Governance concerns.

Each incident is converted to lowercase and evaluated against the keyword lists.

The category with the highest number of keyword matches is selected. Messages without matches are labelled Unclassified. Ties are resolved using the predefined category order.

### Evaluation

The classifier is tested using the same three ESG incidents used in the LLM experiments.

Predicted categories are compared against reference labels to calculate classification accuracy.

### Limitations

Keyword matching cannot reliably interpret context, negation or complex relationships between concepts.

The keyword lists were constructed using terms relevant to the small test dataset, so performance on these examples may overestimate general effectiveness.

The classifier also does not independently assess priority or human-escalation requirements.


### Results

| Incident | Expected Category | Predicted Category | Correct |
|---|---|---|---|
| ESG001 | Environmental | Environmental | Yes |
| ESG002 | Social | Social | Yes |
| ESG003 | Governance | Governance | Yes |

**Classification accuracy: 3/3 (100%)**

### Discussion

The rule-based classifier correctly classified all three ESG incidents using predefined keyword matching.

Its accuracy matched the improved Qwen2.5 prompt on this dataset and exceeded the basic Qwen prompt, which achieved 66.7%.

However, the keyword lists were designed around concepts appearing in the test scenarios. Consequently, the results do not demonstrate that rule-based classification is equally effective on unseen incidents.

The rule-based approach provides transparent and predictable classification logic but has limited ability to interpret context, ambiguity, negation and unfamiliar terminology.

Unlike the improved LLM prompt, the rule-based classifier does not produce incident-specific explanations, priority assessments or escalation recommendations.

A larger, independently labelled dataset would be needed for a more meaningful performance comparison.



In [27]:

# ============================================================
# Experiment 8: Rule-Based ESG Classification
# ============================================================


# A transparent keyword-based baseline for comparison with Qwen.
# The same three test incidents are used for consistency.

esg_keywords = {
    "Environmental": [
        "wastewater", "pollution", "emissions",
        "river", "waste", "contamination"
    ],
    "Social": [
        "unsafe", "working conditions", "overtime",
        "protective equipment", "discrimination",
        "employee safety"
    ],
    "Governance": [
        "conflict of interest", "family member",
        "supplier", "corruption", "bribery",
        "fraud", "undeclared"
    ]
}

def classify_esg_rules(message):
    text = message.lower()

    # Count keyword matches for each ESG category.
    scores = {
        category: sum(
            keyword in text
            for keyword in keywords
        )
        for category, keywords in esg_keywords.items()
    }

    # Return Unclassified when no keywords are detected.
    if max(scores.values()) == 0:
        return "Unclassified", scores

    # Select the category with the highest keyword score.
    # Ties follow the dictionary's category order.
    category = max(scores, key=scores.get)

    return category, scores


rule_based_results = []

for incident in esg_test_data:
    prediction, scores = classify_esg_rules(
        incident["message"]
    )

    correct = prediction == incident["expected_category"]

    rule_based_results.append({
        "id": incident["id"],
        "expected_category": incident["expected_category"],
        "predicted_category": prediction,
        "correct": correct,
        "scores": scores
    })

    print(f"\nIncident: {incident['id']}")
    print(f"Expected: {incident['expected_category']}")
    print(f"Predicted: {prediction}")
    print(f"Keyword scores: {scores}")
    print(f"Correct: {correct}")

accuracy = sum(
    result["correct"] for result in rule_based_results
) / len(rule_based_results)

print(f"\nRule-Based Accuracy: {accuracy:.2%}")



Incident: ESG001
Expected: Environmental
Predicted: Environmental
Keyword scores: {'Environmental': 3, 'Social': 0, 'Governance': 0}
Correct: True

Incident: ESG002
Expected: Social
Predicted: Social
Keyword scores: {'Environmental': 0, 'Social': 4, 'Governance': 0}
Correct: True

Incident: ESG003
Expected: Governance
Predicted: Governance
Keyword scores: {'Environmental': 0, 'Social': 0, 'Governance': 3}
Correct: True

Rule-Based Accuracy: 100.00%



## Experiment 9: Zero-Shot ESG Classification

### Objective

Evaluate a pretrained zero-shot classification model as an alternative approach to ESG incident classification and compare its performance with the Qwen2.5 LLM and rule-based classifier.

### Model Selection

**Model:** facebook/bart-large-mnli

**Framework:** Hugging Face Transformers

**Execution device:** CPU

BART-MNLI is a pretrained natural language inference model that can classify text into candidate categories without additional task-specific training.

This provides a useful baseline for evaluating whether an instruction-following LLM offers advantages over a traditional zero-shot classification approach.

### Methodology

The classifier evaluates each incident against three candidate ESG categories:

- Environmental
- Social
- Governance

The following hypothesis template is used:

`This incident concerns {} issues.`

The model evaluates the relationship between each incident and the candidate category statements.

The category receiving the highest score is selected as the predicted classification.

The same three fictional ESG incidents are used to maintain consistency across experiments.

### Results

| Incident | Expected Category | Predicted Category | Top Score | Correct |
|---|---|---|---|---|
| ESG001 | Environmental | Environmental | 0.9650 | Yes |
| ESG002 | Social | Governance | 0.4694 | No |
| ESG003 | Governance | Governance | 0.7715 | Yes |

**Classification accuracy: 2/3 (66.67%)**

### Detailed Classification Scores

| Incident | Environmental | Social | Governance |
|---|---|---|---|
| ESG001 | 0.9650 | 0.0152 | 0.0198 |
| ESG002 | 0.1220 | 0.4086 | 0.4694 |
| ESG003 | 0.0235 | 0.2050 | 0.7715 |

These scores represent the model's relative preference among the three candidate categories. They should not be interpreted as calibrated probabilities of classification correctness.

### Results Analysis

The zero-shot classifier correctly identified two of the three incidents.

**ESG001 – Environmental**

The model correctly classified the wastewater discharge incident as Environmental, assigning it a score of 0.9650.

This indicates a strong preference for the Environmental category over the other two candidate labels.

**ESG002 – Social**

The model incorrectly classified unsafe working conditions as Governance.

Governance received a score of 0.4694, compared with 0.4086 for Social.

The relatively small difference between these scores suggests that the model had difficulty distinguishing workplace safety concerns from governance-related organisational responsibilities.

**ESG003 – Governance**

The model correctly classified the undisclosed supplier conflict of interest as Governance, with a score of 0.7715.

This demonstrates its ability to recognise governance-related misconduct in the selected scenario.

### Comparison with Previous Experiments

| Classification Approach | Correct Predictions | Accuracy |
|---|---|---|
| Original Qwen2.5 prompt | 2/3 | 66.67% |
| Improved Qwen2.5 prompt | 3/3 | 100% |
| Rule-based classifier | 3/3 | 100% |
| BART-MNLI zero-shot classifier | 2/3 | 66.67% |

The improved Qwen2.5 prompt and rule-based classifier achieved the highest accuracy on the selected test cases.

The zero-shot classifier matched the original Qwen prompt's overall accuracy, although their classification errors occurred on different incidents.

The original Qwen prompt misclassified ESG003, while BART-MNLI misclassified ESG002.

This suggests that different classification approaches may exhibit different weaknesses even when their overall accuracy is identical.

### Strengths

- Does not require task-specific model training.
- Produces classification scores for each candidate category.
- Can evaluate categories without manually defined keyword rules.
- Provides a useful independent baseline for comparison with the LLM.

### Limitations

- Classification performance depends on the wording of candidate labels and the hypothesis template.
- The model misclassified the workplace safety incident.
- Classification scores are not calibrated probabilities of correctness.
- The model does not independently generate incident-specific explanations, priority levels or human-escalation recommendations.
- The experiment uses only three synthetic incidents, limiting the generalisability of the findings.

### Conclusion

The BART-MNLI zero-shot classifier achieved 66.67% accuracy on the three ESG incidents.

Although it correctly identified environmental pollution and governance misconduct, it incorrectly classified workplace safety concerns.

The improved Qwen2.5 prompt performed better on this limited dataset and additionally generated structured explanations, priority levels and escalation decisions.

However, the rule-based classifier also achieved 100% accuracy, so the results do not establish that the LLM is generally superior.

A larger, independently labelled dataset would be required to evaluate classification reliability and generalisation more thoroughly.


In [28]:

# ============================================================
# Experiment 9: Zero-Shot ESG Classification
# ============================================================


# BART-MNLI provides an independent classification approach
# without using the keyword rules or Qwen's ESG prompt.

from transformers import pipeline

zero_shot_classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1
)

candidate_labels = [
    "Environmental",
    "Social",
    "Governance"
]

zero_shot_results = []

for incident in esg_test_data:

    prediction = zero_shot_classifier(
        incident["message"],
        candidate_labels=candidate_labels,
        hypothesis_template="This incident concerns {} issues.",
        multi_label=False
    )

    predicted_category = prediction["labels"][0]
    confidence = prediction["scores"][0]

    correct = (
        predicted_category == incident["expected_category"]
    )

    zero_shot_results.append({
        "id": incident["id"],
        "expected_category": incident["expected_category"],
        "predicted_category": predicted_category,
        "confidence": confidence,
        "correct": correct,
        "all_scores": dict(zip(
            prediction["labels"],
            prediction["scores"]
        ))
    })

    print(f"\nIncident: {incident['id']}")
    print(f"Expected: {incident['expected_category']}")
    print(f"Predicted: {predicted_category}")
    print(f"Confidence score: {confidence:.4f}")
    print(f"All category scores: {zero_shot_results[-1]['all_scores']}")
    print(f"Correct: {correct}")

accuracy = sum(
    result["correct"] for result in zero_shot_results
) / len(zero_shot_results)

print(f"\nZero-Shot Accuracy: {accuracy:.2%}")


Device set to use cpu



Incident: ESG001
Expected: Environmental
Predicted: Environmental
Confidence score: 0.9650
All category scores: {'Environmental': 0.9649760127067566, 'Governance': 0.019799238070845604, 'Social': 0.01522471196949482}
Correct: True

Incident: ESG002
Expected: Social
Predicted: Governance
Confidence score: 0.4694
All category scores: {'Governance': 0.4694333076477051, 'Social': 0.40861162543296814, 'Environmental': 0.12195504456758499}
Correct: False

Incident: ESG003
Expected: Governance
Predicted: Governance
Confidence score: 0.7715
All category scores: {'Governance': 0.7714822888374329, 'Social': 0.20502281188964844, 'Environmental': 0.023494871333241463}
Correct: True

Zero-Shot Accuracy: 66.67%



## Experiment 10: Consolidated ESG Classification Comparison

### Objective

Compare the classification performance of four ESG message triage approaches and evaluate the effect of prompt engineering on the open-source LLM.

The comparison includes:

1. Qwen2.5-1.5B-Instruct with the original basic prompt.
2. Qwen2.5-1.5B-Instruct with the improved structured prompt.
3. A Python rule-based keyword classifier.
4. BART-MNLI zero-shot classification.

### Methodology

Predictions from the previous experiments were retrieved from the stored results and combined into a single Pandas DataFrame.

All four approaches were evaluated using the same three fictional ESG incidents.

Each predicted category was compared with its predefined reference category.

Classification accuracy was calculated using:

Accuracy = (Correct classifications / Total incidents) × 100

### Classification Results

| Incident | Expected | Original Qwen | Improved Qwen | Rule-Based | BART Zero-Shot |
|---|---|---|---|---|---|
| ESG001 | Environmental | Environmental | Environmental | Environmental | Environmental |
| ESG002 | Social | Social | Social | Social | Governance |
| ESG003 | Governance | Social | Governance | Governance | Governance |

### Accuracy Comparison

| Classification Method | Correct | Total | Accuracy |
|---|---|---|---|
| Original Qwen | 2 | 3 | 66.67% |
| Improved Qwen | 3 | 3 | 100.00% |
| Rule-Based | 3 | 3 | 100.00% |
| BART Zero-Shot | 2 | 3 | 66.67% |

### Results Analysis

**1. Original Qwen Prompt**

The original Qwen prompt correctly classified two incidents but incorrectly assigned the governance conflict-of-interest incident to the Social category.

This suggests that broad ESG instructions may be insufficient for distinguishing related ethical and governance concerns.

**2. Improved Qwen Prompt**

The improved prompt correctly classified all three incidents.

Explicit ESG category definitions helped the model distinguish governance misconduct from social concerns.

The improved prompt also generated structured classification fields, incident priorities, explanations and human-escalation recommendations.

However, its raw responses included Markdown code fences, requiring preprocessing before JSON parsing.

**3. Rule-Based Classification**

The rule-based classifier also achieved 100% accuracy.

This demonstrates that simple, transparent keyword matching can perform effectively when incidents contain known terminology.

However, the keyword lists were constructed with knowledge of the test scenarios. Their performance therefore cannot be treated as an independent measure of generalisation.

**4. BART-MNLI Zero-Shot Classification**

BART-MNLI correctly classified two incidents but incorrectly identified the workplace safety incident as Governance.

Its performance demonstrates that zero-shot classification can operate without task-specific training, although its predictions depend on the model's interpretation of the candidate categories.

### Comparative Discussion

The improved Qwen prompt achieved higher classification accuracy than the original prompt and BART-MNLI on the selected incidents.

The improvement from 66.67% to 100% demonstrates a positive result for prompt engineering within this experiment.

However, the rule-based classifier achieved the same accuracy, indicating that the LLM did not demonstrate a classification accuracy advantage over the simpler approach on these examples.

The improved LLM provided additional functionality through explanations, priority assessments and escalation decisions.

These capabilities may offer operational value beyond category classification, although their reliability requires further evaluation.

### Limitations

- Only three fictional incidents were evaluated.
- The dataset does not represent the diversity of real organisational ESG reports.
- The rule-based classifier used keywords selected with knowledge of the test scenarios.
- The evaluation primarily measured category accuracy rather than the correctness of priority and escalation decisions.
- A single deterministic model response was evaluated for each incident and prompt.
- The zero-shot results may change with different candidate labels or hypothesis wording.
- No independent human expert assessment was conducted.

### Conclusion

The experiment demonstrates that more explicit ESG definitions and structured prompt instructions improved Qwen2.5's classification results on the selected incidents.

Both the improved Qwen prompt and rule-based classifier achieved 100% accuracy, while the original Qwen prompt and BART-MNLI achieved 66.67%.

The results support the usefulness of prompt engineering for this limited ESG triage task but do not establish the general superiority of LLM-based classification.

For real-world implementation, a larger independent test dataset, structured output validation and human oversight would be necessary.


In [29]:

# ============================================================
# Experiment 10: Consolidated Classification Comparison
# ============================================================

# Combine predictions from all four experimental approaches
# into one table for analysis and reporting.

import pandas as pd
import json
import re

comparison_rows = []

for incident in esg_test_data:
    incident_id = incident["id"]

    original = next(
        r for r in original_results if r["id"] == incident_id
    )

    improved = next(
        r for r in improved_results if r["id"] == incident_id
    )

    rule = next(
        r for r in rule_based_results if r["id"] == incident_id
    )

    zero_shot = next(
        r for r in zero_shot_results if r["id"] == incident_id
    )

    # Extract the original prompt's reported ESG category.
    original_match = re.search(
        r"ESG Category:\*{0,2}\s*(Environmental|Social|Governance)",
        original["model_response"],
        flags=re.IGNORECASE
    )

    original_category = (
        original_match.group(1).title()
        if original_match else "Unparsed"
    )

    # Parse improved model output after removing Markdown fences.
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        improved["model_response"].strip()
    ).strip()

    improved_data = json.loads(cleaned)

    comparison_rows.append({
        "Incident": incident_id,
        "Expected": incident["expected_category"],
        "Original Qwen": original_category,
        "Improved Qwen": improved_data["category"],
        "Rule-Based": rule["predicted_category"],
        "BART Zero-Shot": zero_shot["predicted_category"]
    })

comparison_df = pd.DataFrame(comparison_rows)

print("Classification Comparison")
display(comparison_df)

# Calculate accuracy directly from the stored predictions.
methods = [
    "Original Qwen",
    "Improved Qwen",
    "Rule-Based",
    "BART Zero-Shot"
]

accuracy_rows = []

for method in methods:
    correct = (
        comparison_df[method] == comparison_df["Expected"]
    ).sum()

    accuracy_rows.append({
        "Method": method,
        "Correct": int(correct),
        "Total": len(comparison_df),
        "Accuracy (%)": round(
            correct / len(comparison_df) * 100, 2
        )
    })

accuracy_df = pd.DataFrame(accuracy_rows)

print("\nAccuracy Summary")
display(accuracy_df)


Classification Comparison


,Incident,Expected,Original Qwen,Improved Qwen,Rule-Based,BART Zero-Shot
0,ESG001,Environmental,Environmental,Environmental,Environmental,Environmental
1,ESG002,Social,Social,Social,Social,Governance
2,ESG003,Governance,Social,Governance,Governance,Governance



Accuracy Summary


,Method,Correct,Total,Accuracy (%)
0,Original Qwen,2,3,66.67
1,Improved Qwen,3,3,100.00
2,Rule-Based,3,3,100.00
3,BART Zero-Shot,2,3,66.67



## Experiment 11: Export and Preserve Experimental Evidence

### Objective

Export and preserve the results of the ESG classification experiments in structured CSV and JSON formats to support reproducibility, transparency and independent verification.

### Methodology

Python was used to export the experimental results collected from the original Qwen prompt, improved Qwen prompt, rule-based classifier and BART-MNLI zero-shot classifier.

Two file formats were used:

- **CSV:** To store classification comparisons and accuracy summaries.
- **JSON:** To preserve original model responses, baseline predictions and validation results.

All exported files were organised in the `Q3_ESG_Results` directory.

### Exported Evidence

The following seven files were successfully generated:

| File | Description |
|---|---|
| accuracy_summary.csv | Classification accuracy for all four approaches |
| classification_comparison.csv | Predicted and expected ESG categories across the three incidents |
| improved_qwen_results.json | Original responses generated using the improved Qwen prompt |
| json_validation_results.json | JSON parsing and schema validation results |
| original_qwen_results.json | Original responses generated using the basic Qwen prompt |
| rule_based_results.json | Keyword-based classification predictions and scores |
| zero_shot_results.json | BART-MNLI predictions and classification scores |

### Observed Results

The execution returned:

**Experimental evidence exported successfully.**

All seven expected files were listed in the output, confirming that the export process completed successfully.

The exported evidence captures the following findings from previous experiments:

| Classification Method | Correct Predictions | Accuracy |
|---|---|---|
| Original Qwen prompt | 2/3 | 66.67% |
| Improved Qwen prompt | 3/3 | 100.00% |
| Rule-based classifier | 3/3 | 100.00% |
| BART-MNLI zero-shot classifier | 2/3 | 66.67% |

The evidence also preserves the JSON validation findings:

- Raw JSON validity: 0/3
- JSON validity after Markdown cleanup: 3/3
- Correct classifications after validation: 3/3

### Reproducibility and Evidence Integrity

The raw model responses were exported without manually correcting or modifying the model's classifications.

This allows the reported findings to be compared with the original generated outputs.

The exported files provide supporting evidence for the classification results and can be uploaded to GitHub alongside the Colab notebook.

Although successful file creation was confirmed, file integrity and archive verification will be addressed in the next experiment.

### Conclusion

Experiment 11 successfully exported seven evidence files containing the ESG classification results, model responses and validation findings.

These files provide a structured record of the practical experiments and support reproducibility, transparency and independent review.


In [30]:

# ============================================================
# Experiment 11: Export Experimental Evidence
# ============================================================


# Export the classification comparison, accuracy summary,
# and original model responses for reproducibility.

import os
import json

output_dir = "Q3_ESG_Results"
os.makedirs(output_dir, exist_ok=True)

# Export consolidated classification predictions.
comparison_df.to_csv(
    f"{output_dir}/classification_comparison.csv",
    index=False
)

# Export accuracy results.
accuracy_df.to_csv(
    f"{output_dir}/accuracy_summary.csv",
    index=False
)

# Preserve raw original-prompt responses.
with open(
    f"{output_dir}/original_qwen_results.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        original_results,
        file,
        indent=4,
        ensure_ascii=False
    )

# Preserve raw improved-prompt responses.
with open(
    f"{output_dir}/improved_qwen_results.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        improved_results,
        file,
        indent=4,
        ensure_ascii=False
    )

# Export baseline results.
with open(
    f"{output_dir}/rule_based_results.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(rule_based_results, file, indent=4)

with open(
    f"{output_dir}/zero_shot_results.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(zero_shot_results, file, indent=4)

# Export JSON validation findings.
with open(
    f"{output_dir}/json_validation_results.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(validation_results, file, indent=4)

print("Experimental evidence exported successfully.\n")

for filename in sorted(os.listdir(output_dir)):
    print(filename)


Experimental evidence exported successfully.

accuracy_summary.csv
classification_comparison.csv
improved_qwen_results.json
json_validation_results.json
original_qwen_results.json
rule_based_results.json
zero_shot_results.json



## Experiment 12: Package and Verify Experimental Evidence

### Objective

Package the exported ESG classification results into a single ZIP archive and verify its integrity before preserving the evidence in the GitHub repository.

### Methodology

Python's `shutil` library was used to create a ZIP archive containing the files exported in Experiment 11.

The `zipfile` library was then used to:

1. Inspect the archive contents.
2. Verify the archive's integrity.
3. Count the included evidence files.
4. Confirm that all seven expected files were present.

The archive was configured for download from Google Colab after successful verification.

### Observed Results

| Verification Item | Result |
|---|---|
| Archive name | Q3_ESG_Experimental_Evidence.zip |
| Archive integrity | Valid |
| Expected evidence files | 7 |
| Files included | 7 |
| Verification status | Passed |

### Verified Archive Contents

The following files were included in the `Q3_ESG_Results` directory:

1. `accuracy_summary.csv`
2. `classification_comparison.csv`
3. `improved_qwen_results.json`
4. `json_validation_results.json`
5. `original_qwen_results.json`
6. `rule_based_results.json`
7. `zero_shot_results.json`

### Discussion

The verification confirmed that the ZIP archive was readable and contained all seven exported evidence files.

Packaging the experimental results provides a convenient way to preserve the raw model responses, classification comparisons and evaluation findings.

The archive can be stored locally and used to organise supporting evidence in the GitHub repository.

### Conclusion

Experiment 12 successfully created and verified the ESG experimental evidence archive.

The archive passed the integrity check and contained all seven expected files, confirming that the experimental outputs were packaged successfully for preservation and further documentation.



In [31]:

# ============================================================
# Experiment 12: Package and Download Experimental Evidence
# ============================================================

# Create a ZIP archive containing the exported ESG results.
# Verify the archive contents before downloading.

import os
import shutil
import zipfile
from google.colab import files

source_folder = "Q3_ESG_Results"
archive_name = "Q3_ESG_Experimental_Evidence"

# Confirm that the evidence directory exists.
if not os.path.isdir(source_folder):
    raise FileNotFoundError(
        f"Evidence directory not found: {source_folder}"
    )

# Package all exported evidence files.
archive_path = shutil.make_archive(
    archive_name,
    "zip",
    root_dir=".",
    base_dir=source_folder
)

# Verify and display the archive contents.
with zipfile.ZipFile(archive_path, "r") as archive:
    file_list = [
        name for name in archive.namelist()
        if not name.endswith("/")
    ]

    archive_valid = archive.testzip() is None

print("Archive:", os.path.basename(archive_path))
print("Archive valid:", archive_valid)
print("Total evidence files:", len(file_list))

print("\nFiles included:")
for filename in sorted(file_list):
    print("-", filename)

# Download only after successful verification.
if archive_valid and len(file_list) == 7:
    files.download(archive_path)
else:
    print("\nArchive verification requires review.")


Archive: Q3_ESG_Experimental_Evidence.zip
Archive valid: True
Total evidence files: 7

Files included:
- Q3_ESG_Results/accuracy_summary.csv
- Q3_ESG_Results/classification_comparison.csv
- Q3_ESG_Results/improved_qwen_results.json
- Q3_ESG_Results/json_validation_results.json
- Q3_ESG_Results/original_qwen_results.json
- Q3_ESG_Results/rule_based_results.json
- Q3_ESG_Results/zero_shot_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## Experiment 13: GitHub Evidence Repository

### Objective

Organise and preserve the ESG classification experiment results in a GitHub repository to support reproducibility, transparency and assessment verification.

### Repository Information

**Repository:** [BUS5001-A3-ESG-Evidence](https://github.com/pragyasrivastava2000/BUS5001-A3-ESG-Evidence)

**Evidence folder:** `Q3_LLM_Triage`

### Methodology

The seven CSV and JSON files generated during Experiment 11 and packaged during Experiment 12 were uploaded to the GitHub repository.

The files were stored individually rather than as a ZIP archive so that the experimental results could be inspected directly.

### Uploaded Evidence

| File | Purpose |
|---|---|
| accuracy_summary.csv | Classification accuracy comparison |
| classification_comparison.csv | Predictions from four classification approaches |
| improved_qwen_results.json | Improved prompt responses |
| json_validation_results.json | JSON validation findings |
| original_qwen_results.json | Original prompt responses |
| rule_based_results.json | Rule-based classifier outputs |
| zero_shot_results.json | BART-MNLI zero-shot classification outputs |

### Observed Results

All seven evidence files were successfully uploaded to the `Q3_LLM_Triage` directory.

The GitHub repository displayed the uploaded files alongside the existing `README.md`.

The repository provides an accessible record of the classification experiments and supporting evaluation data.

### Reproducibility

The uploaded files preserve the model predictions, classification comparisons and validation results.

These files can be reviewed alongside the Google Colab notebook to examine the implementation and reported findings.

The repository also provides version control for the experimental evidence.

### Conclusion

Experiment 13 successfully organised the seven ESG classification evidence files in GitHub.

The repository now contains supporting data for the Q3 practical experiments and comparative evaluation.



### Experiment 14: Hugging Face Spaces — Interactive Zero-Shot ESG Classification

#### Objective

To evaluate the ability of a publicly hosted Hugging Face zero-shot classification model to categorise ESG-related operational incidents without additional task-specific training. The experiment also provides a comparison with the earlier BART-MNLI classification performed in Google Colab.

#### Experimental Setup

| Component | Configuration |
|---|---|
| Platform | Hugging Face Spaces |
| Application | Zero Shot Topic Classification |
| Application URL | https://huggingface.co/spaces/joeddav/zero-shot-demo |
| Selected model | BART-MNLI |
| Classification labels | Environmental, Social, Governance |
| Multiple correct topics | Disabled |
| Test dataset | Three fictional ESG operational incidents |
| Evaluation method | Compare the highest-ranked category with the expected category |

The Hugging Face application was accessed through its browser interface. The predefined example was changed to **Custom**, and each ESG incident was entered into the text field. The three ESG categories were provided as candidate labels, with multiple correct topics disabled to obtain one primary prediction.

#### Test Messages

**ESG001 — Environmental**

> A factory employee reports that untreated wastewater is being discharged into a nearby river.

Expected classification: Environmental.

**ESG002 — Social**

> Several warehouse employees report unsafe working conditions, excessive overtime and a lack of appropriate protective equipment.

Expected classification: Social.

**ESG003 — Governance**

> An employee reports that a senior manager approved payments to a supplier owned by a family member without declaring the conflict of interest.

Expected classification: Governance.

#### Experimental Results

| Incident | Expected category | Predicted category | Environmental score | Social score | Governance score | Result |
|---|---|---|---:|---:|---:|---|
| ESG001 | Environmental | Environmental | 59.4% | 24.3% | 16.3% | Pass |
| ESG002 | Social | Social | 27.2% | 48.0% | 24.7% | Pass |
| ESG003 | Governance | Governance | 10.9% | 37.6% | 51.5% | Pass |

**Classification accuracy:**

Accuracy = (3 correctly classified incidents / 3 total incidents) × 100 = **100%**

All three incidents were assigned their expected primary ESG category.

#### Comparison with the Earlier Google Colab Experiment

The earlier Colab experiment used the Hugging Face `facebook/bart-large-mnli` model through the Transformers zero-shot classification pipeline.

| Incident | Colab BART-MNLI | Hugging Face Space | Expected |
|---|---|---|---|
| ESG001 | Environmental | Environmental | Environmental |
| ESG002 | Governance | Social | Social |
| ESG003 | Governance | Governance | Governance |
| **Accuracy** | **66.67%** | **100%** | — |

The hosted Hugging Face demonstration correctly classified ESG002 as Social, whereas the earlier Colab experiment classified it as Governance.

Although both experiments used BART-MNLI-based classification, the difference cannot be attributed to superior model performance without verifying their exact model versions, hypothesis templates, preprocessing and inference configurations.

This highlights the importance of reproducibility and consistent experimental settings when comparing model outputs across platforms.

#### Critical Evaluation

**1. Classification performance**

The Hugging Face demonstration correctly identified the primary ESG category for all three test incidents. The results suggest that zero-shot classification can support initial ESG incident categorisation without requiring a separately trained ESG-specific classifier.

However, the test dataset contains only three deliberately selected synthetic incidents. The observed 100% accuracy does not establish reliable performance on unseen organisational reports.

**2. Ambiguity and category overlap**

The Governance incident received a Social score of 37.6%, compared with 51.5% for Governance. This suggests that the model also associated the incident with social or organisational concerns.

The Social incident received a top score of only 48.0%, demonstrating that the highest-ranked classification does not necessarily represent a decisive prediction.

Real ESG incidents may involve multiple categories, and forcing a single label could conceal important secondary concerns.

**3. Interpretation of model scores**

The displayed classification scores indicate the model's relative preferences among the supplied candidate labels. They should not be interpreted as independently calibrated probabilities that the predictions are correct.

A low top-label score or small difference between leading categories may justify additional review, but appropriate escalation thresholds would require further testing.

**4. Operational limitations**

The Hugging Face interface returns category rankings and scores rather than the complete structured output needed for operational ESG incident triage.

It does not independently generate all required fields, including urgency, sentiment, follow-up requirements, recommended team, escalation reason, data sensitivity risk and a brief summary.

Additional prompt engineering, structured output validation and workflow integration would therefore be necessary for an operational ESG incident management solution.

**5. Responsible AI considerations**

Incorrect classification could route incidents to the wrong team or delay action on environmental, workplace safety or governance risks.

The experiment used fictional messages to avoid disclosing real employee or organisational information. A production implementation would require appropriate data protection, access controls, logging, monitoring and human oversight.

High-risk incidents should not depend solely on an automated category prediction.

#### Conclusion

The interactive Hugging Face experiment correctly classified all three selected ESG incidents, demonstrating the practical use of zero-shot NLP classification through a cloud-hosted interface.

The results also revealed differences from the earlier Colab baseline, reinforcing the need for reproducible evaluation and careful interpretation of model scores.

Despite the successful classification results, the limited dataset, potential category overlap and absence of complete structured triage output mean that this demonstration should be treated as supplementary experimental evidence rather than proof of production readiness.

#### Evidence and Reproducibility

The following screenshots document the actual model predictions:

- `HF_ESG001_Environmental.png` — Environmental: 59.4%
- `HF_ESG002_Social.png` — Social: 48.0%
- `HF_ESG003_Governance.png` — Governance: 51.5%

**GitHub evidence:** [Hugging Face Experiment Screenshots](https://github.com/pragyasrivastava2000/BUS5001-A3-ESG-Evidence/tree/main/Q3_LLM_Triage/HuggingFace_Screenshots)

**Source application:** [Hugging Face Zero Shot Topic Classification](https://huggingface.co/spaces/joeddav/zero-shot-demo)



In [33]:

# ESG incident test dataset

incidents = [
    {
        "id": "ESG001",
        "message": "A factory employee reports that untreated wastewater is being discharged into a nearby river.",
        "expected_category": "Environmental"
    },
    {
        "id": "ESG002",
        "message": "Several warehouse employees report unsafe working conditions, excessive overtime and a lack of appropriate protective equipment.",
        "expected_category": "Social"
    },
    {
        "id": "ESG003",
        "message": "An employee reports that a senior manager approved payments to a supplier owned by a family member without declaring the conflict of interest.",
        "expected_category": "Governance"
    }
]

print(f"Loaded {len(incidents)} ESG test incidents.")


Loaded 3 ESG test incidents.


In [34]:

# ============================================================
# Experiment 15: Eight-Field Structured ESG Triage
# ============================================================

import json
import re
import torch

triage_prompt = """You are an ESG incident triage assistant.

Analyse the employee message and return exactly one JSON object
with these eight fields:

1. issue_category: Environmental, Social, or Governance
2. urgency: LOW, MEDIUM, HIGH, or CRITICAL
3. sentiment: POSITIVE, NEUTRAL, or NEGATIVE
4. followup_required: Y or N
5. recommended_team: appropriate organisational team
6. escalation_reason: brief explanation of escalation need
7. data_sensitivity_risk: LOW, MEDIUM, or HIGH
8. brief_summary: concise factual summary

Classification guidance:
- Environmental: pollution, emissions, waste, water, energy
- Social: employee safety, labour conditions, human rights
- Governance: corruption, conflicts of interest, compliance

Assess urgency based on potential impact and immediacy.
Use CRITICAL only for an immediate, severe threat.
Treat unverified allegations as reports, not established facts.
Escalate potentially serious incidents for human review.

Return valid JSON only, without Markdown fences.

Employee message:
{message}
"""

experiment15_results = []

for incident in incidents:
    prompt = triage_prompt.replace("{message}", incident["message"])

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        generated,
        flags=re.IGNORECASE
    ).strip()

    try:
        parsed = json.loads(cleaned)
        json_valid = True
    except json.JSONDecodeError:
        parsed = None
        json_valid = False

    experiment15_results.append({
        "incident_id": incident["id"],
        "expected_category": incident["expected_category"],
        "raw_output": generated,
        "json_valid_after_cleanup": json_valid,
        "parsed_output": parsed
    })

    print(f"\n{'=' * 65}")
    print(f"Incident: {incident['id']}")
    print(f"Expected category: {incident['expected_category']}")
    print(f"Valid JSON after cleanup: {json_valid}")
    print("Model output:")
    print(generated)

print("\nExperiment 15 inference completed.")



Incident: ESG001
Expected category: Environmental
Valid JSON after cleanup: True
Model output:
```json
{
  "issue_category": "Environmental",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Environment Department",
  "escalation_reason": "Potential environmental contamination with high urgency.",
  "data_sensitivity_risk": "MEDIUM",
  "brief_summary": "Untreated wastewater from a factory is being discharged into a nearby river."
}
```

Incident: ESG002
Expected category: Social
Valid JSON after cleanup: True
Model output:
```json
{
  "issue_category": "Environmental",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "HR Department",
  "escalation_reason": "Immediate concern regarding employee safety and working conditions.",
  "data_sensitivity_risk": "MEDIUM",
  "brief_summary": "Warehouse employees report unsafe working conditions, excessive overtime, and inadequate protective equipme


### Experiment 15: Eight-Field Structured ESG Incident Triage

#### Objective

This experiment evaluates whether Qwen2.5-1.5B-Instruct can generate structured ESG incident assessments containing eight operational fields. The evaluation focuses on classification accuracy, JSON formatting, field completeness, escalation decisions and potential operational risks.

#### Experimental Configuration

| Component | Configuration |
|---|---|
| Model | Qwen/Qwen2.5-1.5B-Instruct |
| Environment | Google Colab |
| Task | ESG incident classification and structured information extraction |
| Test dataset | Three synthetic ESG incident messages |
| Prompt strategy | Explicit ESG definitions, field requirements and escalation guidance |
| Generation method | Deterministic inference (`do_sample=False`) |
| Output format | JSON object with eight specified fields |
| Validation | JSON parsing after removal of Markdown code fences |

The prompt defined Environmental, Social and Governance categories and required the model to generate the following fields:

1. `issue_category`
2. `urgency`
3. `sentiment`
4. `followup_required`
5. `recommended_team`
6. `escalation_reason`
7. `data_sensitivity_risk`
8. `brief_summary`

The instructions also required factual summaries, human review of serious incidents and appropriate treatment of unverified allegations.

#### Experimental Results

| Incident | Expected category | Predicted category | Urgency | Sentiment | Follow-up | Recommended team | Data sensitivity | Category result |
|---|---|---|---|---|---|---|---|---|
| ESG001 | Environmental | Environmental | HIGH | NEGATIVE | Y | Environment Department | MEDIUM | Pass |
| ESG002 | Social | Environmental | HIGH | NEGATIVE | Y | HR Department | MEDIUM | Fail |
| ESG003 | Governance | Governance | CRITICAL | NEGATIVE | Y | Compliance Team | HIGH | Pass |

**ESG category accuracy:** 2/3 = 66.67%.

**Field completeness:** All eight required keys were present in each of the three generated responses.

**JSON parsing:** All three responses were successfully parsed after removing Markdown code fences.

**Raw output formatting:** All three responses included Markdown code fences despite the instruction to return JSON only.

#### Observed Model Behaviour

**ESG001 — Environmental incident**

The model correctly classified untreated wastewater discharge as Environmental and assigned HIGH urgency. It recommended the Environment Department and required follow-up.

The escalation explanation referred to potential environmental contamination. This was consistent with the reported incident, although the appropriate operational severity would depend on verified circumstances and applicable response procedures.

**ESG002 — Social incident**

The model incorrectly assigned the Environmental category to a report concerning unsafe working conditions, excessive overtime and inadequate protective equipment.

However, the recommended team was HR Department, and the escalation reason correctly identified employee safety and working conditions.

This demonstrates an internal inconsistency: the structured category was incorrect even though other generated fields reflected the incident's social and labour-related nature.

Such inconsistencies could cause misrouting if downstream systems rely primarily on the `issue_category` field.

**ESG003 — Governance incident**

The model correctly identified the undisclosed supplier conflict of interest as Governance and recommended the Compliance Team.

It assigned CRITICAL urgency and HIGH data sensitivity risk.

The escalation decision is defensible as a precautionary response to a potentially serious governance allegation. However, the incident description does not establish an immediate severe threat. The CRITICAL urgency classification may therefore represent over-escalation under the prompt's own severity guidance.

The generated summary also presented the reported conduct relatively directly. In an operational setting, summaries should consistently preserve the distinction between an allegation and a verified finding.

#### Structured Output Evaluation

The model produced all eight requested fields in each response, demonstrating its ability to follow a defined information-extraction schema.

Nevertheless, field completeness did not guarantee semantic accuracy or consistency.

The ESG002 result illustrates this limitation: the model generated a complete JSON object but selected the wrong primary ESG category.

All three outputs were enclosed in Markdown code fences. Although the cleanup procedure allowed successful JSON parsing, the raw responses did not fully comply with the JSON-only instruction.

A production implementation would require schema validation, enumeration checks, cross-field consistency checks and controlled error handling before outputs were accepted by downstream services.

#### Critical Evaluation

**Classification accuracy:** Two of the three ESG categories were correct. The misclassification of ESG002 demonstrates that explicit category definitions do not eliminate model errors.

**Cross-field consistency:** ESG002 contained a mismatch between the Environmental category and the HR recommendation. This indicates that field-level validation alone would be insufficient.

**Escalation risk:** ESG003 received CRITICAL urgency despite limited evidence of an immediate threat. Over-escalation can increase review workloads, while under-escalation could delay responses to serious incidents.

**Data protection:** The model assigned MEDIUM sensitivity to ESG001 and ESG002 and HIGH sensitivity to ESG003. These values are model-generated assessments rather than verified privacy classifications. Real reports may contain employee identities, commercially sensitive information or allegations requiring additional safeguards.

**Hallucination and factual reliability:** The outputs were broadly grounded in the supplied incident descriptions, but the classification error and severity judgement demonstrate the need for human review and independent validation.

**Reproducibility:** Deterministic generation settings reduced sampling variability. However, reproducibility also depends on the model version, software environment and inference configuration.

**Generalisation:** Three synthetic incidents are insufficient to establish production-level accuracy or reliability.

#### Comparison with Previous Experiments

| Experiment | Output structure | ESG category accuracy |
|---|---|---:|
| Experiment 5 — Basic Qwen prompt | Simplified classification output | 66.67% |
| Experiment 6 — Improved Qwen prompt | Four-field structured response | 100% |
| Experiment 15 — Eight-field Qwen prompt | Eight-field structured response | 66.67% |

The eight-field prompt produced more operational information than the earlier four-field prompt, but category accuracy decreased on the same three incidents.

This result does not establish that more detailed prompts generally reduce accuracy. It demonstrates that prompt modifications can change model behaviour and require renewed validation.

#### Recommendations

1. Introduce cross-field consistency checks to detect contradictions between ESG categories, recommended teams and escalation reasons.
2. Validate JSON structure and allowed values before storing or routing incidents.
3. Apply clearly defined severity rules to distinguish HIGH from CRITICAL urgency.
4. Preserve allegation language in summaries until incidents are independently verified.
5. Route uncertain, sensitive and high-impact cases to authorised human reviewers.
6. Expand testing to include ambiguous, multi-category and adversarial ESG messages before considering operational deployment.

#### Conclusion

Qwen2.5-1.5B-Instruct generated complete eight-field ESG triage records for all three test incidents, but classification accuracy was limited to 66.67%.

The experiment identified a significant cross-field inconsistency in ESG002, potential over-escalation in ESG003 and repeated non-compliance with the raw JSON-only formatting requirement.

These findings demonstrate that complete structured output does not necessarily imply reliable operational decision-making. Additional validation, prompt refinement, risk controls and human oversight would be necessary before limited organisational deployment.


In [35]:

# ============================================================
# Experiment 16: JSON Schema and Consistency Validation
# ============================================================

import json
import pandas as pd

required_fields = {
    "issue_category",
    "urgency",
    "sentiment",
    "followup_required",
    "recommended_team",
    "escalation_reason",
    "data_sensitivity_risk",
    "brief_summary"
}

allowed_values = {
    "issue_category": {"Environmental", "Social", "Governance"},
    "urgency": {"LOW", "MEDIUM", "HIGH", "CRITICAL"},
    "sentiment": {"POSITIVE", "NEUTRAL", "NEGATIVE"},
    "followup_required": {"Y", "N"},
    "data_sensitivity_risk": {"LOW", "MEDIUM", "HIGH"}
}

expected_teams = {
    "Environmental": {"environment", "sustainability"},
    "Social": {"hr", "human resources", "people", "safety"},
    "Governance": {"compliance", "legal", "ethics", "audit"}
}

validation_results = []

for result in experiment15_results:
    data = result["parsed_output"]
    errors = []

    if not isinstance(data, dict):
        errors.append("Invalid JSON object")
    else:
        missing = required_fields - set(data)
        extra = set(data) - required_fields

        if missing:
            errors.append(f"Missing fields: {sorted(missing)}")
        if extra:
            errors.append(f"Unexpected fields: {sorted(extra)}")

        for field, valid_options in allowed_values.items():
            if data.get(field) not in valid_options:
                errors.append(f"Invalid {field}")

        for field in [
            "recommended_team",
            "escalation_reason",
            "brief_summary"
        ]:
            if not isinstance(data.get(field), str) or not data[field].strip():
                errors.append(f"Missing or invalid text: {field}")

    category = data.get("issue_category") if isinstance(data, dict) else None
    team = str(data.get("recommended_team", "")).lower() if isinstance(data, dict) else ""

    team_consistent = (
        any(term in team for term in expected_teams.get(category, set()))
        if category in expected_teams
        else False
    )

    category_correct = category == result["expected_category"]

    validation_results.append({
        "incident_id": result["incident_id"],
        "schema_valid": len(errors) == 0,
        "category_correct": category_correct,
        "team_category_consistent": team_consistent,
        "validation_errors": "; ".join(errors) if errors else "None"
    })

validation_df = pd.DataFrame(validation_results)

print("EXPERIMENT 16 — VALIDATION RESULTS")
print(validation_df.to_string(index=False))

print("\nSUMMARY")
print(f"Schema-valid records: {validation_df['schema_valid'].sum()}/3")
print(f"Correct ESG categories: {validation_df['category_correct'].sum()}/3")
print(f"Category/team consistency: {validation_df['team_category_consistent'].sum()}/3")


EXPERIMENT 16 — VALIDATION RESULTS
incident_id  schema_valid  category_correct  team_category_consistent validation_errors
     ESG001          True              True                      True              None
     ESG002          True             False                     False              None
     ESG003          True              True                      True              None

SUMMARY
Schema-valid records: 3/3
Correct ESG categories: 2/3
Category/team consistency: 2/3



### Experiment 16: Automated JSON Schema and Cross-Field Consistency Validation

#### Objective

This experiment evaluates the structural validity, classification accuracy and internal consistency of the eight-field ESG triage responses generated by Qwen2.5-1.5B-Instruct in Experiment 15.

The objective is to determine whether structurally valid JSON records also contain reliable information for operational incident routing.

#### Validation Methodology

A Python-based validation procedure was applied to the three generated ESG incident records.

The validation assessed four aspects:

1. **Schema compliance:** Presence of the eight required fields, absence of unexpected fields, permitted categorical values and non-empty text fields.
2. **Classification accuracy:** Agreement between the generated ESG category and the expected category.
3. **Cross-field consistency:** Alignment between the generated ESG category and the recommended organisational team.
4. **Validation errors:** Identification of structural or value-related errors in each record.

The category/team consistency check used predefined keyword mappings:

| ESG category | Accepted team keywords |
|---|---|
| Environmental | environment, sustainability |
| Social | hr, human resources, people, safety |
| Governance | compliance, legal, ethics, audit |

This approach provides a basic consistency check rather than a comprehensive semantic assessment of organisational routing.

#### Experimental Results

| Incident | Schema valid | Category correct | Category/team consistent | Validation errors |
|---|---|---|---|---|
| ESG001 | True | True | True | None |
| ESG002 | True | False | False | None |
| ESG003 | True | True | True | None |

#### Validation Summary

| Evaluation metric | Result | Percentage |
|---|---|---|
| Schema-valid records | 3/3 | 100% |
| Correct ESG classifications | 2/3 | 66.67% |
| Category/team consistency | 2/3 | 66.67% |

All three records passed the implemented schema validation checks.

However, only two records contained the correct ESG category, and only two demonstrated consistency between the generated category and recommended team.

#### Analysis of Findings

**ESG001 — Environmental**

The generated category was Environmental, and the recommended team was Environment Department.

The record passed all implemented checks, demonstrating agreement between the expected classification, generated category and operational routing recommendation.

**ESG002 — Social**

The model generated Environmental as the primary category despite the incident describing unsafe working conditions, excessive overtime and inadequate protective equipment.

The recommended team was HR Department, which was appropriate for the reported workplace concerns but inconsistent with the generated Environmental category.

The record passed schema validation because all required fields and permitted values were present.

This demonstrates that syntactic correctness does not guarantee semantic accuracy.

A downstream workflow relying solely on the generated category could incorrectly route the incident, even though the recommended team suggested a more appropriate response.

**ESG003 — Governance**

The generated category was Governance, and the recommended team was Compliance Team.

The record passed the implemented schema, classification and team consistency checks.

However, successful validation does not establish that every other generated field, including urgency and data sensitivity, is operationally appropriate.

#### Critical Evaluation

**Structural validity versus semantic reliability**

All records passed schema validation, but ESG002 failed both classification accuracy and cross-field consistency checks.

This demonstrates that JSON validation alone cannot establish the reliability of AI-generated operational decisions.

**Cross-field validation**

The keyword-based consistency check successfully identified the mismatch in ESG002.

However, this method is dependent on predefined team names and may incorrectly reject valid organisational team assignments that use different terminology.

It may also accept incorrect recommendations if a recognised keyword appears in an inappropriate context.

**Operational risk**

Incorrect ESG categorisation may lead to incident misrouting, delayed investigation or incomplete reporting.

A complete validation framework should combine schema checks with semantic consistency assessment, confidence-based review and human oversight.

**Validation limitations**

The implemented schema checks verify field presence, allowed categorical values and non-empty text fields. They do not constitute comprehensive JSON Schema validation.

The experiment also does not independently validate the factual accuracy of summaries, the appropriateness of urgency classifications or the correctness of data sensitivity assessments.

The small synthetic dataset limits conclusions about general reliability.

#### Recommendations

1. Apply automated structural validation before accepting generated ESG incident records.
2. Introduce cross-field consistency checks for categories, recommended teams and escalation decisions.
3. Flag contradictory or uncertain outputs for human review.
4. Use organisation-specific routing rules rather than relying exclusively on model-generated team recommendations.
5. Extend validation to urgency, sentiment, sensitivity and factual consistency.
6. Evaluate the workflow using a larger dataset containing ambiguous and multi-category incidents.

#### Conclusion

Experiment 16 confirmed that all three Qwen-generated records satisfied the implemented eight-field schema checks.

However, ESG classification accuracy and category/team consistency were both 66.67%.

The incorrect Environmental classification and inconsistent HR recommendation in ESG002 demonstrate that structurally valid AI-generated JSON can still contain operationally significant errors.

The results support combining structured output validation, cross-field consistency checks and human oversight before using LLM-generated ESG assessments in automated organisational workflows.


In [36]:

# ============================================================
# Experiment 17: Export ESG Triage and Validation Evidence
# ============================================================

import json
import os
import zipfile

output_dir = "Q3_ESG_Structured_Triage_Evidence"
os.makedirs(output_dir, exist_ok=True)

# Preserve the original model responses and parsed records.
triage_path = os.path.join(
    output_dir, "experiment15_structured_triage.json"
)

with open(triage_path, "w", encoding="utf-8") as file:
    json.dump(experiment15_results, file, indent=2, ensure_ascii=False)

# Preserve the automated validation results.
validation_json_path = os.path.join(
    output_dir, "experiment16_validation_results.json"
)

with open(validation_json_path, "w", encoding="utf-8") as file:
    json.dump(validation_results, file, indent=2, ensure_ascii=False)

validation_csv_path = os.path.join(
    output_dir, "experiment16_validation_summary.csv"
)

validation_df.to_csv(validation_csv_path, index=False)

# Package the evidence files for repository upload.
zip_path = "Q3_ESG_Structured_Triage_Evidence.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in sorted(os.listdir(output_dir)):
        archive.write(
            os.path.join(output_dir, filename),
            arcname=filename
        )

# Verify archive integrity and display the exported files.
with zipfile.ZipFile(zip_path, "r") as archive:
    integrity_result = archive.testzip()
    archived_files = archive.namelist()

print("EXPERIMENT 17 — EVIDENCE EXPORT")
print(f"Output directory: {output_dir}")
print(f"Files exported: {len(archived_files)}")

for filename in archived_files:
    print(f" - {filename}")

print(f"Archive: {zip_path}")
print(f"Archive integrity: {'PASS' if integrity_result is None else 'FAIL'}")


EXPERIMENT 17 — EVIDENCE EXPORT
Output directory: Q3_ESG_Structured_Triage_Evidence
Files exported: 3
 - experiment15_structured_triage.json
 - experiment16_validation_results.json
 - experiment16_validation_summary.csv
Archive: Q3_ESG_Structured_Triage_Evidence.zip
Archive integrity: PASS



### Experiment 17: Export and Preservation of Structured ESG Triage Evidence

#### Objective

This experiment preserves the structured ESG triage outputs and automated validation results generated in Experiments 15 and 16. The objective is to maintain reproducible evidence for subsequent analysis, independent review and repository-based documentation.

#### Methodology

Python's `json`, `os` and `zipfile` libraries were used alongside Pandas to export the experimental results.

The export process involved:

1. Creating a dedicated evidence directory.
2. Saving the original model responses and parsed structured outputs in JSON format.
3. Exporting the automated validation results in JSON and CSV formats.
4. Packaging the three evidence files into a compressed ZIP archive.
5. Performing an archive integrity check using `ZipFile.testzip()`.

#### Export Results

| Evidence file | Contents |
|---|---|
| `experiment15_structured_triage.json` | Original Qwen responses, parsed JSON records and incident identifiers |
| `experiment16_validation_results.json` | Schema validation, classification accuracy and cross-field consistency results |
| `experiment16_validation_summary.csv` | Tabular validation results for comparison and analysis |

**Output directory:** `Q3_ESG_Structured_Triage_Evidence`

**Archive:** `Q3_ESG_Structured_Triage_Evidence.zip`

**Files exported:** 3

**Archive integrity:** PASS

#### Evidence Integrity and Reproducibility

The archive integrity check completed successfully, indicating that no corrupted archive member was detected.

Preserving the raw model responses alongside parsed outputs allows subsequent reviewers to distinguish between the model's original formatting and the results of JSON cleanup.

The validation files provide a separate record of structural compliance, ESG category accuracy and category/team consistency.

The successful archive integrity check does not independently establish the semantic correctness of the model's predictions or guarantee long-term reproducibility across different model and software versions.

#### Conclusion

Experiment 17 successfully exported and packaged the evidence from Experiments 15 and 16.

Three evidence files were generated, and the ZIP archive passed its integrity check.

The exported records support traceability, independent verification and further evaluation of the ESG triage workflow.
